# Independent verification of g₃(7) = 474 (Erdős Problem #817), self-contained

This version needs **no GitHub access and no token**. It contains compressed copies of the verification program,
the search program and the published count table, taken from commit `19aa62291988` of the repository. Cell 1 unpacks them
and prints their SHA-256, which you can compare with the files in the repository. A CPU runtime is enough.

| step | what it checks | time on a standard Colab CPU runtime |
| --- | --- | --- |
| 2. quick | certificates (from the definition), Lemma 1, g₃(5) = 60, g₃(6) = 168, exhaustive search at N = 473 and 474 | ~10–15 min |
| 3. critical | exhaustive search for **every** N = 419…478, compared with the published count table | a few hours |
| 4. full (optional) | every N = 1…478 (does not rely on Korsky's bound g₃(7) ≥ 419) | +1–2 h |
| 5. Rust (optional) | the independent Rust implementation at N = 473, 474 | ~15 min |

Each value of N is logged separately. If the runtime disconnects, run cell 1 again, then the same step; finished
values are skipped, as long as the logs are on Google Drive (step 3).

## 1. Unpack the programs and data

In [ ]:
import base64, gzip, hashlib, os
ROOT = '/content/g3verify'
FILES = {
  'Makefile': ('032b9a9ded651546904429a1a9b0f25217f49791fa04adb77b9b77f5b1f8f059', (
    'H4sIAAAAAAACA5VU224TMRB9jr/iSK2qRGQ3atOqsKWCNlwl2lSBB3hKHcfJWtm1F9ubKhIfz3gvpYEgwstqx545M3M8Zw5wXaps'
    'Dp5lKKxZWp67GJjI76WyMpfauwQcIwiTFyqTFt2lEDAWIuN6eYHlcMGdP0HppINPJa5vPp7g7u3XL1DaeVsKr4zGQyo1O0CUcyvS'
    'S829WktIzWcZhSkPrqkGjZB2zTNKi8JYH64p81yi4D6FIXz7oJzsQ8bLGIR7Nbnp9anASek8vDGZSLkKmbqC26XpYUGlhrKUnstC'
    '0oeg19KqhZK2X6W92/iUkIaPriKVYuViNhrh1SWoXTZ69+nq/edgRePhdhOMEXUJZkoPWiqe/E+1scbkzZGTIbA2VtNtq6pp0zjS'
    'SyyI7MaqrxgjK2GdfDVXFlERLquzNlcCZ0VrxAI/Ko/OYXc06oG+VQ89RAaHr3H4ciu2qXNfiDe348l4fPMHVt1Si1Jbe1fSMtKE'
    't+Z/xNdEJfX7bn6d7E9GQ3zbQWPuH/97Bc3B1LpBQMxpNmPrdl2PwrjG3uRZm0vMd/jh6AjVZGNW6TaKrMwkd5L8i124npylHzRe'
    'j1dUNiOVhIcmgWmnnJdabJrZR5fPTOlxglzp0kvXSzCXC6VVJeY1rYh2WfRpNwQ7CLAPIa0naQlOMX2stHkgb56V0rEKeU+hPBXD'
    'bqGwzoy7FJTGu4Et9bQRrUtDW0/EHrVip23j7YbWitI+2SoUzyrZz6VQLqylumDc4hKn58M+fU7RPYuOzx7ZuKActPBwX1S7Y9gS'
    '39IuXZn5uCA6LTEmeHaPbmpK63q0o2BlRBVDkvemznL8Io5Pz5+zZnxY51+4tCbFijFBjxp2gs0R2UVg6O8jwFh892F8+y2pdn1F'
    'V+OMCob9BAaF3JsLBgAA'
  )),
  'src/g3fast2.c': ('f9f991d637075bb89937375e81877d19167f7d5198f5051a9d7d3c626f7eccda', (
    'H4sIAAAAAAACA7Vaf3PaSNL+35+ib1PZkgw4IDtOzhhfJXFStXUEp5zkbuvl5SgZBqO1kChJ2DiJ77Pf0z0jaQTC8V7VsRsjRj3d'
    'Pd1P/5iRXuzv0T5dH878NPMOJtRqUbzMgkXwTU3J+fT+9y808aNpMPUzRSkuU5fUeu6v0iy4xYjyk8mcZnFC1+NDJ3LJeZ9M45Q+'
    'JfFVqBb07HXnVZMu3v/2md4c/vWvr7y2ewCJLPSzv1C08LO5wp9gkhLkUMqDhmuWKEV+Cu30ANRzUgxhBk3ixUJFWco/EuWeMEOi'
    'N+RPF0GaBpBNdNo7I4pi/B99U0lMEwoi+t7yDg68h39FdBdkc0pXC5qMA/Lxr0dtzeZ83MeP73xz/D047fUfCpqTKpc+PTQJQqdB'
    'dE1rClIK1bUfUjCb4WcUZ6Blbrw0zxpgG9CXOa9/HSygw4DnLkN/ArPPgiTNurLMmJdHCpaUxfoJVj6PUxWxEkE0SZSfsug4maok'
    'N+w5pMMo0USlNEviRcWAS5XAWwsfdymOwnst5zZIgwyitcmhrLbxKhcaTCE/mPhhbumWLAs6Mw/2okoCPwxS8IgjmX0XRNP4juIZ'
    'LWMwD+KIneVnjCcCesKQrhRhAVMKebphzJ9Q3aqQopZHkVLTlIatQZOOBqOuudMvx1ueA7pW3xWKRvFrROQEEdimasILE2dDrVJI'
    'lqyAtNVyGScZOHn77O1+k8zFSHAqK70KslTBEBM/Se5JoLQ0Lk9jWWs6D2YZhXG81Oa6SmDfeWvG1nT8VRa3bqFGnLB9CvwP4mlu'
    '4xMgY6rS4c0IuItWiys4HYazwHzTEhW+Dx5oRW/J+fH2B0hvWh1XLw04wvBpjwYtmOCm0XFZhMPKnX/4rP2bInL9SQZ/YThV7gHR'
    'BwTuDcvEEJwZSDzVaxBpDQppg3wdq9S/Vid5DgGrQRjTYB7QMM3i5djPxgLoXnvEI2rZ6+Ains3ATsYWQcQAx3DpHFomMeCbAsED'
    'qAeODT2jaV03mFuTDg4OZOHzwJpvmNIZdYCxNEuCiU4WmyGVxYUG4N2Cbg4woZLWFTwzNalIaZyzTV/sPUPchSv47jTNpkF8MD+r'
    'DoXB1eZYAqxs0QGe1TGkXcUje8+magaV6OOb3wfUae9l90uFIVphyvHROKPV8VF3b+/FPiEcojHj01m7J7K8Q0/wys5bE6KNKcoI'
    'bAK5kxvEAzhp6IaI0bkfzgCGr2zutx9/80jy/t0cM/1bPwh9+L+7xzGDCRwvPEC3CC7wRE7mnA0jr0K4x6eZuqNFzEkjwspXExHs'
    'MrZWqYnEeJVpQY46uD6gN5cf3SZymNxjOa3zwcVYlHC4tLz79DVldcBTBsFqEUyAEATNVJcNLKOJaEbVQMF48/Gc/g/Kd154ALm4'
    'bEbaplNnPGbJ47FLv/5Kf8lHjTzXckewAC7YdeyTNOMahSWF7BjY3zI9/1q7qBeJylZJROOlWmdjjDrrJrXXLzc+X/t9t0sPe89U'
    'mKonMWaTwJu/9uq5dc39Hjlr+sF/zgB7FwsE/eHGZxe9Z+jbH6r/7aI/yunbHz7Y/3bRvy7omYpp9fdO/Y+tCTlxZYKx9rq7x7ZE'
    'jzLbK62ZUYRw7toDl12q+yCK8vCgJUMLWSthWGUcSJg5VWvcaNAlOY088buMKov3pzfn/+zW8zaV4g4lmrMIKR8gTVFQN1h8ufiy'
    'i8X8PuXaa3ggOZk6WDJgpOyfX61mQ8kZDeqMKmu/C+PyThOxFFiEWgozJ9Sta0XzOBSd0fT5XcZjco/ygAFGLJtIamBVfa7IVSE8'
    'siHHtrWISoUK1q5bFLPlvFyzpjCGNvJHF05D4dVSpHFYMYaUJd1klbn2n7+d/+4s0b/igy+422UYHvMle9ctKN+WlDnhr3R86O5t'
    'xDKLulYZQORMkAEz46Sm3FjmYW1QzM2K6zjnQ6PHiIUbScy/s3IZ55zzp2k2DOODg3nA3YKTJhMEDf6envr64uzMz0e8Ysjz3XxW'
    'l0cKNyIXCxq5aLDDDQZKN9zGwRSdwxJ5VjLSfl5PWZMmWWsrboC/Xqavv8JYf8+DfNnihw70B0Xq4dtD7fa7xb07vgcCdkGTrswv'
    'tjPQy/SYZO6ZX3zPzJ+RgxlI71deLo8/XErYzsStfhh38Y2+YR7gotGwCfnDC7plzslkCJNV7t3SD234YYD28A49w+kpdHQ5eZWj'
    '+NMRNzpg1eL77m42DWED4gobHhXgs4CnsIFcT2vjVbXxNrXxfqKNp7Xxqtp4m9pssWF0CjBvy/EHuXrQ6eN/5Q94/G+Iyf/CKy52'
    'dda0n/H+U64qeTd28faqej/dfxW9vZ/y/lNOrei9yfsRH5sclbeg5dYPrb+fIF3IVyNUkcv3z6l1xhlo2D44QMnAMEPg+Gjfa1cK'
    'gSRUSUR6W1mTUQ1/STcKxV/ugbWdce7aUFoyrFCj6bzCwNtyoExA0R0nV1aowcldko2VXnbmlbbAGNMNinlxOlGfDyG+QYH2gbgg'
    'H9p0QB5Uj4fMTlGFJMOlWFSCHVFPrFzNlhjX07FqwRlaTZQ3mNAFxqGXEOBO1/bvufYp190MzuNeiYZt8aKb+3WkbZg71X3Uq2Nu'
    'ff8L1/LvbLEcvvRG9T2edB6WHkewQufl4TFrGQH/GD56jfauQ9crH20Q90KiaokGrjIOF6k6SBhQnjdzVQ1hk9VySzszG9RyV7SN'
    'OBzZi0/A3KN+L+HBhsDmu2ftIVgSaxOMrETNdPOgnk6waNEWqAJjQBbTgIdDz/2fYas8b1z46Y0sPcKGKt+y0zfB2VY6OZF2HWBR'
    'mRy/fevpe+g6sCX81vS+5cdv2xhk/Qux6VMRyDNsCKqhdzR6ije34CI8hZ1NoMPhXKPJplRWnprgXwGiJ8KEJWmf/tvJr39A/QpG'
    '2Hdg0yPtN26nnNy7KA/5vE1v2jTi1EIo1Gr0aDy+WgUh/DBexks5/QrDXIkKpkxjjGkCDLsZTRQfQshC77VTil0yN/uNhuazxN49'
    'mzm/fL7of/3y28WABr3nU/r+CzaGbo29OqYBiaD3obFZzuL5tIlpshkp1LTuyf8P/x+BBgqtLQGzcJXOUVymnK7shbDU6SwV6X2D'
    'LJ/76XLjkq4W+bJsRHKC551ef1QGWV8cBcWl4X0WzPi4aHBxeXHxsXQom4u906OBeLQlErvmIGIHmWeRyR7bxofQnXJCM95qV3PM'
    '/aQMiYK3RqwVbkA4S+DE0xTR+DtxN2bpaLqvC6eKJ+8N9O8Z+zwLuBWRZ+Lhe3Er9r+hArfJ8B7u3Og0hQurWEJ1kn0DSg15tRfS'
    'gxwG5qoC+oIfp0azRtYffmQNGxBUJdbb2b4k4RzHNqOq/+53HmmsJUneNzrmUNPUMts5/kS8IzWq6qafZ7D8U2aye+0+8R44u90t'
    'lf5gx2wqUviN77a8Lr4Rgvy9vQXIUfcHgx2tBIICzlmp7hbVhlqw1B9s8OZmTa7VpC53sq04SfbEWJv7j4e9JzPaXhG7YpZwLtZC'
    'urVrfiQTY/KTk3BhIB0As6ROo9z9dTGAGd3aCVoL/K2Vl7PkQ757Owg4BrId9Fi2OT8xNRBZ/scP2hjjMKgdb+jxvR0dYQ0f/t7F'
    'S997DHTV8PW2wjf/mOrFhcLdvfLykKpIrjV2fdiBw/Kqmpgr7ZofRHyopwuHOVSVEoP9XVFkzLmbfYL+7uuXiw8fxE7l8TmXGnfP'
    'hLqmONFHh/rxXtG+LVbg7odpTGHAnZd+1LhKEmU3YdLGOef68eJ1Et+l2Kzpp2350ipN4do8EvQ5RXKiLdbX6li1huXxqTznxIOc'
    'kVb+RD8kMzpbD75YE/a6MdeAuF2JIzmRNY9I/zG+cQZQkEN/ksRp2prM1eQm1U8fCvA/Xnw3SEoH1RRgRsh6R/GVpwaTai+6frzo'
    'riUvridWX7nWyX/96G5kbVLcmnPcmpPc2q6v0GJdra8yqS6vGNIS4XqAc4q5quSVPJds1tQ155N11s1bIA1Ei2mUtVq79ogb9cpC'
    'jT7WBBL9q/hWQbJVxcQVbN7Tissqbql2WA43se5aOtuNpmpX+ZfOs88d+7octDpFPhhbLXjtiJKlH5kD1bSOeCk7xBaTNWkp20C+'
    'rnb+THSaH+prfVwzszK4MQvczvKD/2KWiKgMVh0pXOVURk7BIQWhJJOswXlggQPeuVFqKakjiG79JPCj7KR6nj0M4yYfkVuuqlTm'
    'u3INuu+/K9TjCsuIljNJxnNx4NXu1jPjbTHD8MxiS38jM3pijbpb0h6RYTHTJ6SWjnJSWgJEn8/Lify5bEH4zL1iNHYQ7zj6JuaB'
    'FHeztlS2Xm3Zr/ASGdayVj+5nmCnOvcT2t/Hj9vidA3M+SasdsSPQmdmd4Sdj0qSJv1i3hJ4nm69ICBFzrwbULwZMJLNFEsYttlm'
    'RiWva1RkhPtZHDhCUhxYsI4DwVN50xsBTgPBUzl4mM8oVeD7vIIzOoLrStKjER+GWsc0rGhJ/LJC/FKIYV69jJLsuEJ2bPMEmvmd'
    'qzjyw5PiZQWUyLB8S0G/tJDNEdkDfmXhSvF+Jn/JwbmC71v5TJW/tdDUD9y5hO5ZoLVfb5B9Zf6MXr8KZL3dlawieWgv40in/irM'
    'CqE9TnFWdSvHzYpfVVb8Shum3LLm9FLBLKbWaRmDibuMCNz4CV8tsORVqOERNuAaM0K4DRhmOJD0xAd/tZz4tnQnsO6pOSB0zGtM'
    '6IVmKkldiKhhfmnybcSv9eig917rhfCzw/zocMDPv15wpWrQkb7NT33z25c8ryDwTNMpj5arZTfUVTdkLXm5uOQMIg1bKGfN0sa5'
    'fhhcozsbA0rxxDk+QsQH31Q8k50Cvx7kiPgGvWKprzHy2rVPRsyrNyDQaO4CfPpNG75q9CQSKgdLkHMzxs6Bz9nlh2MloIVagIkj'
    'Na5J7UIbGbCfI/Ehzo5ku2Pp33PmxgY2e7NYXipcxylVbKQf2XNClZ9iiZZ2F7caVguN+MyfUm88vLZqi6mKcgbD/FpQzdS8Yow7'
    'lEHNovpUnDr1Oa6xwH6rxavKa+1os+6a3UoppCApK0QjJ3mo1ABD1B4hFi45vFqGsx6pDY1LJJ2Y0gWQVB8B/BHbt0f6EXbble2y'
    'vSt9q8c3upo2K25ZRbdAUuCshq+6S91qZrx+n5ea9zPm58aJxmZbs9XSbD6f22ppttqZTs2pyZ9vZ6wCLjbktyeMMZu8qK1SnvcG'
    'lb5gqyfY6lI05CudjZG31XlszdVZTc+2YuUnbEpwcOeBspgn+hbXyIG1pGm84s2W4gB39A/XMSkE1FmbU9S7/sW7v38ef3p/Of78'
    '/l05OT+KlQPeiP8gg6ykqPWeh+FUg+rkF7Emul5OMG5NIE71MfBUjoHxbR8BEzMCB43Pqb1XKUj4Db7e8wNvJpVIVWRsnATX7/W5'
    '8xTl6CpR/s2upuw/IEN9xrYtAAA='
  )),
  'verify/check_set.py': ('3295335908da5d2b032807f3d13bea9a08b371bac844e729b5ac5b8e60f2f4f6', (
    'H4sIAAAAAAACA21UbW/iRhD+7l8x51OK3TMmQK65cscH615EpDRFIu2XKGct9hBWh9fu7pqEJvnvnVnbBNJDApvZ2Zlnnnlm3r4Z'
    '1EYPllINUG2h2tl1qcae7/tetsbsR2rQxtUO+n3AB5FZyFBbuZKZsAjOA1alBgGZULnM2Uo3IIHgq85LA3NdLjdYwNsPw/Mw9rzP'
    'fMVEcC/tuo0olcU7pBiabAVamQHFAlWCWZfaZrU1Ew8gECHYNcL4uwJTFwb4N5WA9BX0hQApFDyeRsNo9PxdhUARESoh9b00CLk0'
    'VqrMfuRYyyaWqZeEtk+BHOpZkIQwhUcX+VFwuMUz1TaBResK5Ypqe4asVFZIZRilKhX9NVYoC+O+RV1AMqck/Alq+ARb+t43Fdfw'
    'jl6nMNqGUUMf5rDcgUUGdwe4Rb1zmOkqsRRywjY1Fx17TAPTwyVwffhPLbdig5Sc4lxiUQgY8q28zMzgj6/Xsz+/LOIi/wjLkgDw'
    'lS6vVDlWSD/KbnbUm9qIO5x0GoCj/g+j8WgYnY3fR2fn59HZh9+is99H0fvTM4CbotSu6wbiOL718EESWCtsbeAU5GrVVsU1VMIY'
    'NA0UF58qYq3JoqJWgyT2bFluTGcwO+N5Xo4rYkgroXdp10Zq1cSR7LQwhRt6kgJ+pXaxIDEC179/ZRVgRYJLwrA5qAzb95niSpd5'
    'TfGC04iqhBE1RhMtwk6JVMoS3ro0Gm2tFbCNM5JOps0ftI2B7u0PW8xN41I2pWNRpSuNuMc9I9B8OXT/GJpmYFqoOwya1CSWYevd'
    '+SyOwWdlQaMrrCQBBgkhP3B3SWKR54wpWIRNohmTZYhbzINZZ6LONGBm5gWOjEiulA1VXaCmyebTYzT3fD4zN5KRTm6Pc8sVy/8d'
    'i/gERszXaaPc1jgYkNXdR3t884Dvb2JjsGGWskduKg4D8Iwc9uda14fuV6XCthkFzWvQ4i9/pGKzoZrZfV+w0HeMhzQX0+v25qii'
    'hDVGqyqwjY54b/GNmMSyERkGvcdeBL1e+GJ4bg2m2kgb9KJeyJzY2FhNqmx11TLVSYll+qZRFvWf8hTSvX0i9o5JqjSj8S+u/k4u'
    'L77AxdX8r2ufdX7kta/UEXl0xDtMqvrFaNPyR0S/mWI5/GTg9p5r57lmT7fKFRonq5/rfX+tXQpT8OfJYuE7hXBSJwuOGQISSvC/'
    'JReXvveq0hPattOTnBr5wI9kSoanDmazBjqsE6CzgJyaUX1qNvu4n8z7jKk5fpo9UZzwxPivxHcCQYM0avsQcU73bAeHXw/Yio4I'
    '+Z+UwXeldjxJ42TZlro3J3NC5XPy1hKF4Qt1bR9/mb5mrNmBJFleuwHv287XxR/yLiJbmipRYJryGPppysOQpn6jqGYyvP8AJxtV'
    'WgsIAAA='
  )),
  'verify/bruteforce.py': ('6b85f5d48302fff80a6356892ea8edea62b4dacb28537cc29e76e926d23baf03', (
    'H4sIAAAAAAACA61WW1PbRhR+9684dYap1NgGm9B23DgzTptMMhNMJ1D6wDDKWlrsBWnl7q5MHMp/73dWkiUDafNQDcjS2XP9zk3P'
    'vtsvrNmfK70v9ZpWG7fM9WGn2+125qZw8io3sRysNtTvUyKvlFZO5bqfyrVMyXOQZ+E72UykKcXCSkuB0olcSdy0o/yK3BI8wjqy'
    'Uph4KW046HT+zM2NpUQZGbt0Q1cmz+iNSXL7PeVGLZQWacsqBf6Mfjf5PJUZPft5+FM47hCud8E0pAndkS2y6E6Q0nR6T4LGdArS'
    '3Ervw5TuwRsIi5PTN2dMUtrJhTQ29GqmpCwt8jwhejl5VamNc+2E0pZ0jj+NV+sEgjrsO2kyEka5ZSadimll8oWR1rKvBb2kNf5v'
    'e3h8TrfwbrQedN47SnLAMzs5o8JKD8vdQW/YG933lb4GDmqt3IYMI58VqeDAf4Gb1kmRkHIE7GKA5pbC7XIB2ZUwwsl00wmctC6S'
    'fxVqLVKpYxmSzb0xd5tDBfhieK+slwQeRhI4VQLxhMSC43UkRbykHFIGuXqLBHvCjL2QusgkG7M0/fCBdL9E2TKmF8PBYHZZ46b0'
    'AiJCJ/B2lRuwLPNbyoTeeKuMdo9WacEhyU4sALGKkfc4LzSYz6ObYBZSAHtzaVi9z8/NfxgMfUHeAHWc6R7dLhVczwpEJZAj2blF'
    '1siaeH9xyGU5GsSQz1YqRfz+rP/b7OTjycmxd91T1gDsagOB8iEyFrAUVizkuG4c2m0aTbNMfKbmClZGcVS5lpQq3FaIaVZ6yaxl'
    'GeJ6Wl+/30qp12fgHZrmOcnPS4Ho1FpSm4frIPTtrDJGH7mTxuV5amtCqaF+sxvb6XTQc1XbRGgoiy4o24xfuM0O7v0rQ+x7bTqu'
    '/S5Z/gaPhVOiHAvMwvRSykhXmJJQmVoKGx2KVQCnCwyGUhl8PjOFJHVVDQ+eApK4lT+VjJ+a1tx2W5mpVsdx6KwOIuy6RZAyqS2V'
    'J54uHRMrUgYK8GtROA7FcQCuhQyysIm4ADMzXqjLLY3Zrxt2BX+GPWpL8XVbS15f7tDVFQU+hJD2aESTCR34KqyJ+/ugQvna7upr'
    'ocvQtdF+CzOygpsbKJpvomaybhNccevcbZOyWwdhpQNVpIXZRPXMkk2NSKkrQBvoJLu7Lb0BBmVSxC4IDnqMyyjs8XCQwk0YdVhp'
    '1RPrKrJAKvqBhCq1qR4/QuUXtQpkj9ivFnhlwcGPXXh2oNjqB9tAJElQJbqNXx3q7igNnFFQMTk8OID7kE8mw9Hhi6MaQoyfSdVV'
    'g4/+J2CmUv0cQxzp9M/PWk07Jl6dD2ba8MUlP1r1RfKAeLGF86YpLcB31IKLT6e7YGOsYb2Xgz7YCg2PAPrNg4JUXykO+m7yLxlv'
    'Xxzfc4yzRwd+8AXd4/enx9OzX991OWkVCtUMS4XBJubCsdtAoybQEvWWSR7uAHvA4qx71KMfmzI4rk7jZa6Qs4sRcnXE5cZJQ+ou'
    'G9ZpxWpFtkplA9Exd61HqfP/APQkOF8Fpjp4OM0pU/jOcvwJNQYzdO4Uri8wrrCyeMstGgHLaBZgD86a6Vpu1t52SY6/vln9x0Y1'
    'bLGugvnGj+QGgnA7Z8+h7aIcZzb3I7d6e1y32uPbQBRvO6Pmf/0tlVzG0N9RVaf1NUwEs174TVX+uJrjp2sZCm4YZP1Yoo57IFb8'
    '8RtMG9PnNS3eSdh5zwtU+cqAcFB5AjM8DrGSB2iNdUivsNH8IqhJF8NL9qO7813QbbzKObeP5lfjEuuRn5ULDtgYuCVGI4D0DDzF'
    'uQJbxsoD/03z4Gx02Qz72U6WPfduoquQoeNxee7kvxzkYB0/1TKnJx/+OHt/MqPZZC+huz1738W+RLqp2+sOrnMgmfECcwb2wrC1'
    'IyoFXk7z7XxMe5YtFb64QKpVwakuPdR2HqLmfG7gW1sx42FdkhducIVP2mXA+xLQRpEWmYwin60o4ixHUZWoMuWdfwDXzKPLig0A'
    'AA=='
  )),
  'verify/verify_result.py': ('c196ecf0eac69335c83a262da254eb1ad3f463a0201afa2cb80d07f17db3e702', (
    'H4sIAAAAAAACA61abXPbxhH+zl9xtYcDwIEgUtSLzYTtuLaSuHEoj+V6OqVZDAgcSUR4YXGAJI7H/e19du8AAiTt5EM8Ywrg3e3t'
    '67O7d3z6l9NKFaeLODuV2b3YbMt1no16T5486d3LIl5u/UKqKim9zVacnIg8kydhnqZBFgkej8OgjPNM5Eux8kf2lSMm4vzqXCzz'
    'QgRiXWGmmSgLYQfZVqRBuI4zKR7ici1WYXgaJkG26hHFd7y7GH0vgkTloqgyJaosXGNcRiLOxE95vkqkeJUnwcIVSkpNe3sa0je+'
    'fvHizTZbOF6vV6lgJcc9UYtVzz6Q7L9VHN6Jzj/7fxcnwwuRxpnzBwiERVxCFYmYnZz8li+U+MecHvOqFK/fvJ8TvXVeFWospqSg'
    '4QvPO796/kcoL6skYYb+EOVhQzeIIhisqFQpyrzRpyjXEoqM5EbiIyvFe5rQGCgoNX9XI5etaGdSRkqEQbHKnV6P1TQmZhZVnGCA'
    'qG2KfFUEqXJFuJbhnRKLHIYNZVFq75BK2FcnSpba4mnwSKRd8bz73XB0+dwRUVzIsEwgdpGnRL63s0gkl3EWk7e5ooAb6t3eSnij'
    'GIqsSiEFTJBsaRhcRVWIvckrL8grLweCfIzeL+l9ePlc2MxrRqNCPpaFTIOktSMYVI7Ly9gXSVz5uA6gs/heYjgownVLadAXVB2l'
    'sVLxAn7KYju8fmriwpaPAQtItBBNtIfjtvZEdG2CIs5WPIOtH+ZVRkYKy7xQWmOs+GqRxGqNyCgD2k37jDrNrnxeobxQ3Xu92jXH'
    'X2GfIlXCA7Yd1xQ2zS4o8kSWl+ABU7DVYit+ARd3W4vVkUNlrtgklSLpeGVLloc11giVJxVZTWHzWEFaIyOoJQQEIJk0gHBMMNJf'
    'TQRTFSTrkWLYE1u8Dw3nUQ67E9OFhKKxpuF4Ab1EQIb3ZE1wwDpLaZOxIOfiTe+DpAIBINpU2GSiJF9hFIxsECJTwiGOP4cp6H1I'
    'oqqUkde7foxLocqghEoGIl4uDYPsrmITKEWzCF/jdJMXJYisoAwl6/cwz8KqKBCc3rIqK3AoAoTgshlX9/XjKskX9XOu6qeiIaWq'
    'BeIAUdCMqW3zWMap7PXe39x8gOpy5W2Ccu0h/rIglfbX3oOFor+275NCfN9xnN6Hl39/e92i8VseZzbRdYVlfNLCY8ctLaf39s30'
    '2n+FdYX0SH2gZxfWf6YT+1P0nSMy87fxnvr7PJIAO3s2OHkh5viC5Jjwq4fXmvD73yX8sU1kb5eayu0Bldubt//88OZmKmpynz4z'
    'FRdUPn3BOiMxFs7mvV4PoEVoBIXbpEhX5HcukKwM4mRiWc6Yo8Ws8YINwbLdzHQcHt4ADkrb6p+cK9FXfWWJvrCtdy9vby04GOYJ'
    'mSgprB9fvnkLTevVtiUoYVviO7OdQ3P1o5lvIRSXCN315ENRScdwq9Z2mEauePbs7qHhD36YtdzJAxzqWeFDNNG2DoMNeauPyEAo'
    'MElXlEBV88j0zB7hpvKXSbBSttkB8WDfQ/a8AEtBGhOIrwPlL9L4zNHJ4JT2PsXKOFvm3wt7itB0xe6T5auy4B4CMmzY0lt5VG7c'
    '3DoeBRxtVBbbcYNQRIkcF1q3rQ59y/EKGUS2NoB8DOWmFDe310WRF7v1Ri/7nPC4lkb7j0ZauI/+0o+jT+rZGP/tT7dwNpcZ0Vtp'
    '4feWgSkz0KyL9teRNveW8Xe0wvaedWcbxo3KvVWRVxt7yBo0fLOHaJHI+fTuzUSeaVjdzex16ieLbGcRWjIfzVpPbZK4tA0NZrsh'
    'UfsHlxd2CWikDDxuKXTnHkQeArdcydiqLAJ8b1n8upNogu9eVsgvGdLhy19fW5xXan1PxNloZ9iGiDh5Pb3x313/64PVFu6pAAHx'
    'b5mJ4enZWNC4iBWKxRA+BICKhK2S/MH5XlSQjHMaEIC9koYF4aQ2GxgLtRKgqJl1cjMSJynZb5IFlKUpfpkbIKgehMDHx5rXectD'
    'IQQCemalwZ0kFD5hLH7149uXP91OaIXefY6nWt278AB0edpVmGsoaTDuGNkgE5mrBDTpFG6IP+mrJ8IGt2MaMXruK224SV85GsaE'
    'pRNGvbtrOHJrg/fEn/qv7T57+NeeZiKEBloYXHiqjABwqDDpSRZFJ55+RJktjQsneRD5bHK7i6KfmdDMmlpzh1KQef3Yj0ghd3OH'
    'neKO/IHLL3uIWtmZ6zjExCZVYX2XZ1pX0Dqq+17HYfkeECYLm/GNs7TjfDHsNTWW39Cr+cQXhCSfvzQeuiSiVGx49GH/fqJXYZDR'
    '4zMPtZPlODuvIXJc7YEi87V0ui6VYmude700KIFjNLsrJ/wyHR+4BbEN7INVlgFYsUld6Q6xqF0k1PHQGDXfnzkd+xGJWj9Ukvng'
    '3iZRGwgK3Vo9GhuZZu+oYK11HbleteUSWNWkjOOyYVMs1X7yqJ3jkTapZThvENX4SD0wcti/82D31YXTcpn0rGbp/VFVEytnf5CX'
    's2bPfWbOGjbBzcAbtPYffdvUtP+ouz/bmA04Om7BxkJ16VVlPtRrT1FOUdhSszqxwrrwIhvtV648z+orvx+x9xJK0TLkdrMX1Vxm'
    'BXc0Stua88me38wGc0oM1CSQmQ9LB5d5cBloDsHqKRp3qkO2IqJGxOZ2xahoEWdBQZWC5fHxzWq0DFR55md5kecpV4bENue90DIl'
    'Xz1VHzPoFFmmG1BhVXyHGZCg1AOM5uzLmAKdPFgOtSLLnRR7FeFM84SpV4h+VRb21Kn/ztu1okbRydKcGbTwF4pFuZwEodSbshZ7'
    'xzTWBlum4mu4nCpjazotcXULiTXkFAf2x26UGNHlKG13tqeftzlaBChzy4GrTUBlvcs9h0cfNjn1Tlnh0vuwJoO9y/Pk+lGGFVp2'
    'Ow0e/Ye8uJOFmhBTrEX52ALFSjcM8tGDRlOEj/FbV7QdVwccN6AK5YSMWNaC+kslNcdzdpqEcqkqdRGCByyEhGUHhJcVZ4qlhyqb'
    '+ptEEkHiZA+QG4Vr54vA6LLyNN7bXWC2P8K00I7ToOReOHRmf4RO1QCT2ESz6bwziq4GSPOR3PejPrlR9Kz0sz0VP/B5CiTRW020'
    'iT0UERTtGuz3EiRb8LuJGB4rYiDppB9xgwVJ+tFpH3bve4MlnQEifIINOnddtIC+ld91m69f39z++vLDq5+tYwWLJWZGe3NeVavS'
    '9GHat1yRINSmVALZLQcTJ/A+R5yKy8E36hUQJZDJ7w4TIxy47iynnVDCgImfNADwGbsHhAb1oYT3sliB06x8R2+FsXewIRD2AzNm'
    'WynKQ4uCOY8BBpOZxSeFVAHUp0/0TGc2KFqOk9Cnm5hWbjdyAotQm8yZfIIopSqfTw9sTpnDrxJBsFidld+qVOrD1iqznK9SpCNU'
    'TA5CKpImlkJES7+EASyzhNS18bSnYyE1Ir2WSXQvM7NaGE17H4HsudPOD3xmwKV1ZBm0I33mVRKZEzk+lBCqCGtCXmjtvEJtFaWn'
    '0jbKagMhWG7Xp+7xclCL8VQMvc5xrnt4Sts6m22KIczLuKmxrdHgzD0fvHDPz6/c8wv8vbzA/0sXAQzRrsjhrecvzt3h8MWlOxwN'
    'Rvg4H+Ljgp6wYDi6HNDHc4tr4cP+RgtLeKultOp7AU4MAANvs6WUNHcONNwSTWSEAJ/76guHeYYFjnvYA+kDYdMK4A8SpiL4N2cy'
    'zgECtOYW8cZ2ZuPh2cCw8lScec0hNhI8VJdWCV+qIE+sEJh0gs9nsQ/7av496RdFVRLBUGrx4c0SkXkfAGhCWYdirYiaidYUAXOT'
    '81tprLhEo6O3ATf1u1bI3hdPl4BUzCkdrjMLe81Oho3II0/cZflDVp+1tg7p3b0Tes5cSHqcX4PmoO571kn35F6f1zcOCPOtY1en'
    'Pk4Iuse2L1zehv7zmcPvdpj2pUu8mI/P1nBw5Q7PL9wheeblGf7Dddk3jwxd8tCXY07bLdxMxZQ5RMa8rGOn5bFLOr2mOuGwt2k1'
    'NCx7ylV5sLHb9bXbcsSWhfggJt1lX868CxQwNm+o61rkQORcrj9c4fOpjh5EQGgTcVrWWlasWFj+T+jdPz/Wuz42uxLD02bnL1wX'
    'dE4uapcmV+oTkwJx3VwKRXV083JHn8lamMHc03XORDfjNrK0vdvoiPh/9ukEWZmZqiPl3DP4GsaKLm1MwHRv6cAWIwQiF/GKuoIr'
    '4aZZCbzUQJfJyztXnHLdWVPSPiCT7pomhe8to5sYW9fc58MXROCFYZvKmq9PHranmuLr4FrK5di+cslw7QsZhoE+91lBksiEq3xx'
    '8ldhjsWbEirwdP0feHRV07bTfgmFMojOEfdaCF63I9POnl3YzCZXIgyyPOP7X33FQRiKl2OXWU0JP+P6kgpCsM3hRWyjV6AHPHHt'
    'gN6j7WEdHNYSE/ewF/3R1WSjMBo16mhcgdwFe0/VzjzfKto79ZPRiBX6dNWnD3YOIm46weBY1LecBAP7F6KA0G8UBF8OC2iu7of1'
    'NSC/ff5mTfHlmOtz9dlqXUHJbxB1urML3JXh5Qd9pdtt6r+ikN1RwdShnn82nIu/oFKYH+jnyAWx/q1Ett3taZu6CkCK2pSO7skZ'
    'dhzr5NCSAAQsy2lJTUXruH2S0q1CTfs/32v14ETFXijssMEVR4Xn8tg5Hih1v63n7J2van1847cIq0JKtf+TBKsJjGJHsIs3bfJd'
    'kgfFNP2ugek1t0Z/s0zZi/6b8iBlADyQiXxKFFz66A6iA2CfMsQdEAqf5ocJ5qrX4BKR0w1evRr7cy+oN5p0xkwoC3F78+u1ePXz'
    '9atfbgVd712/7ph5D9fZWY/R2ynHsDvNSzk2P3thIoZpQqx2QUwE6bvyIT/IQYTHnvjQ/EohCeJU6B+MWHshbNWX9NrBx/xTlCd1'
    'YnlC7Xzz0wPYmUFeX9p/am7tycmf0E6YDuhHpHnGp5pOZ/BtfQ7pegkzfJ+uSH2fNef71Pz6voEG3Qn3/g8t3BIOFyUAAA=='
  )),
  'results/n7_counts.csv': ('e54df8b158490ee84f3932d449caec038681f979db50d1c953af4ed5de0e6369', (
    'H4sIAAAAAAACA2WcSbJsOY+c57GWGLABu01omHuQWZlqUNL+5Z+ziZv2W1pmvncInsMGBBwOMP7X95/8/ad8/6nff+L7T/v+07//'
    'jO///Pd//b//+7//+//8zyd/8zf9/edT/uNJ1ZPyryfxH0+ansS/nvT/eDL0pP/ryfSTP9/7LP1NT/+8/JOTn/W/jxi2npe/L8vl'
    'PGx/HzJ4tdR/dY/9sIy/D5mCWnr8fdj3w/Z3jJmJqGX+6+tzPxz/+jrToSXNvyuc7tO/nyrMiab419NynmpW5ffUW6Km0b/r9zTu'
    'U00tfo+ZmP5e9Joyf4/7fpzX/Nbxe8zk1Fa0DiXq7/k8z0Nz/PMWZqi2qhnm1X86k/bzoreX34w+lVmqsXb9236freU81yfL+s2p'
    'MlM1hgY/128FauznVc9i5N9zJqvG0OaM8mc8/Tzvld34zasyX7W2YJHjzxfmaUCFUvozBaas1o4G5fmbA2+goU1J1P77RjBptXYN'
    'qpSfLn2inAatREn9N+1g2modTSu+/nw84jRorUpffxp8AuM79W+U9qehn4ZUtcA/RfwEM1fr0tJGiz8fn7thau+i/X0VM6eV17T6'
    '510t7ZYlhQyW8jUwdVpR9tH+7HgrpyV11vLP9xuTpxmFGuvPZrW4LdLOUVf7tTB9mjXF74q/Lf20lC4zksqft7EANEeiOcWfwc3T'
    'VPWnxX9eC2tAc0Pjy+i/pp5OU0zMS+e83SaWgfbuEz7Ss4SfXk5Tk+lQLynLuk0sBO0D1RohXRu3KU5T11Q1lqHT+rqxFghMXrs4'
    'UncCn95vG/ZujOWR3kZb6I5J0NA1Vq/nbZy3sVqCnu01siqSKJlD0frUll51/ox0GpOWoUh5lk7Ba2RlBudCcyyjaKRt1NtYbmPC'
    'ro+pL7f+WlkebF2V1tUUiYOan7eJ2yqTWDSi8Z0z30aWiDVrmmNtHbcxrqZ/Rj+toflWaYYV4Y2YVRpYSGxYX7IPpaY3qHlbM9ZM'
    'uiPrOq4eDNZJImVKe0JGXWdyXBPymem0Dn1Mz7HM+S0GSoXPKatjxwbb0Fd5vrTcZq1VJLYg5nVmn8laSaTq6HxjDZkLnYW78TNO'
    'a5IKxKi8K6VyV2uyWpKpGraO0sJVpXzP2Idd2c1ailhJs/2dwM9kuSRSQ0ZIi+K1mf29e57mOjmm7G6uke9iTwOCib+QSkUrGMp5'
    'jcVnpdPctJI9cfDKqtfLfdDRr2S0ljKyqy1tlUb2epfbLG/c9W4pSinXm3wWa7bwMdroMYa2UsOL1xynGdWSGuBiV3rnhXOpddAu'
    'S7Fn6XI0kbQ4t7mf5qQNHzMnvbxoEW4zqyaZqDi+WTWW1ke+27nmaS4yujOFrHX8rP5nsWq23jqOSydRH9PLr7sQpEpHgE2ZK6Sr'
    'TW6rv/YNsKRIA9Ekr1W+c+hvT6JciV75RmgDViuzPQEjL4kF781p9bWN/fWxH1zCFQGkyDZr36ec4XwSBmUJALZtNI5YJqCt/ET6'
    'FUkbaVUOmUxN/83GgE2CDe+gU9f5nBT5WvwPAO2IFCCHPoSzm+1nbTQDi8g+2fRLEcAOM3p+IoaqFgmAjoAPOEgfyj/k6pVlvQfm'
    'VcsGstNRf1ufN5BFpIMDasUn4APmm9GGtRJstsZN9hRjNtJ4M9ogF5GJz5J6SE1llVDmK+LVBfPm7ZvQ5DIrDVekXxEOrj7UJwZe'
    'jvw3Fq+uBHvFjsgto5iyP+knMq9I4VD1lLVHmnSrbxs3VOYcctLzlC2RRdHqpqe0Gzf7qDLp0bI2R1/L+WmUQTSCUjP7Pv4c+nN6'
    'q2tEbZEB+pipASNlKefbxrKDBqGshEGVQdOfe5v5p5gljkhfyx9qWE6pZfzCEa9uAaiABLSB+l+bCTtyRfoVKeA52S/Zu7banw0w'
    'HkdQUBBHKJXElGlCP5F5RUDCxcurD0lf3uoapiMoS0NwoPkuWZ8U5RcopSsytFFFG63907HOF41LxKsrQTXhj6Noj7TP67mZbByP'
    'yFjysToASSBllcj9DdeQHkEpGr5gBGPRls63ukb3FtGy6UNgH8Tjz1i8unxfM5UuSZU4t/KH5WmDQb9l7Bi6AoLmWf95jZdXghOF'
    'kWslWJECKNb4ycwrg9sss7CDGXT1W2AHBUgujr9OUMFcyBRpei/qTFcmZXxFr/SSeWg/jXCogOSq9qhaWs7U0GF543HUYBkBBSKv'
    'yvYWLeHzLtkBBJKrN3yvENTgyAjGvFPpWMIyMog498VMu5ayvaMQOyqO79K64YVr+HS28bOd0a/MJBIULOfrTWHLzwI42LCk9icM'
    'bsBcMgcp/T42nxDGQrDblm/qYPwMqIMQiybssDwjBkNuu6afHXA8soUCoSFEFaBT7fJba8cmFtUxl5nQN2UpBRB0PH6sQHlCfWFu'
    'hNAlpHePnyF1xGLRbJdYKxOuackJ/T4XVyhpoXD8lRi76oU/q+04xqLaYOIyIXt0RdP7KZtDmi2EPeVz0g4pwJoPAmRHNxbNw256'
    'pYVGyaX9tsVxzhZqsn8RdjlFQLvF73NecUQLSyz8N9QoNFEUPzyiJF2hjKrK/g8WnrP24Fh2GGTRgk1saa0GPu5/bZUDoi2E0go+'
    'AItq9/l9Ql5xRIWscOz6s7yO/ErPbzEdJG2htj/HVIX2ZLjetjhasqiBtWAQi8mwx8/fOmzaQlP2S1CR3ZGV+AWhEtq8EMGCgfLo'
    'xPrAn9p/s5tPiD/Lk8rGablk0OPEixLyiiNq5W4MhklqhO6/Waj0hAT0wROzG/LNnssT8oojqv2SF1fQIs3spdpvjSNUntDMfE4m'
    'neWa+tqJRj/Z0ZZFg8GgtNo7gZ8FezGPUDwh9FPfqRvqJEBBud/zkiMb3Y44ONU665rf+Nb7rv6kBKBwRVGIr+UBhbr75eK86MgG'
    '2zcEImGoCMtyPmTeJzsq21LYqN7HxI1qjLXCFxwpLzuyJjcE0BaerAsiEIf0szsO07ZYXsSnJTUzMpOTVcclBL30CDciRcEVWejv'
    '1AhSOLw+YuWJNUEGhSIwHkOvZD6HLZKYlx/hru/JjyvkgZpYo2yTcubgUM5yrJQgwRqZgBQzCY12dtMhnYU7xg4frt1cGL/Erl6x'
    '/sSKXIHOu6IvQXRpEWb6J+ddQLrDVczaR3IMKSXXQq24o5tPjrUS+JKZUAAiR6kTOcud7OZNJTwAd5JhHWRcdd4we2uc9610Bbvm'
    'qe8Wr4oM/Qqfw3rkvBdrsyh8cNjACKh0YoFyPJUEyxOs0hCdqMloZasFIRYBxH2j9wNxqRIk2izmb+QgcbEC7WdpHCZuwcFc5Pcw'
    'qKnp0ArmyYYefXfAaPEJCMFxT1uACTLH1t/Vduy4JbVv+naDrZXH0IbA8rV56WdvC+KTg7ZwXs1chrRLZ2jMs+COJ7cg67OEqbCw'
    'ZQiS4hPTHFfSW4M8GBR3LtDbOKPyv2Eg1w7Lna7kNNqU/5l+52is2DjYU4Kb+QZGm4lLPVA2q5yJLZDfES1PtNgn6/+EBTWk6Cxg'
    'qrUeUXPkdFhjB6ABraB9F+rF6PfYxItE44liyzSADZdak4JA0ej5OqIm09VB4skapA9V9JTQtAEx6hXtTzSFWefR2aYmC9CB0oaT'
    'W9SkuzroXwYjW9wJZbSygrc4XiH7+9r5ZMOHonSYTZ3jMThTUoZ252VyHnRkxhbltNrjTYRBZWnzOSDFkatFZcUqsZr8EsizS7cE'
    'xF9oKVFvF+gthzMTstvacy2ITJHspSLoHeJLtDzRgqOS9iwUSDF6FaKBEOl3AN6uTambip+KHQZ0eV2OH6WQ7SyBQ9st27H/8jLN'
    'uqvYVgZMI3hK6BCXDgrhmHotAG5em8yOCVn3O6/+RE0HlKljvzz0QmJA+DhtTuFTHPOWGyf5RCUdNq1DsWWRLuZ5ROcT5RluyR+R'
    'NevAzhlj3gyQ90tCpZoEkCmGlNb3ZfAmvKCR+s4LpSsr3AdPqwCUGFJWqBE06ijVM7GdWSIGrc5y6BxM/tZBuqYPNOoz2p1vsqxZ'
    'wCqoyjZI1RrmRHNo42zDyUJBsa7tWmM4q6Y1TY5nc2v3vfFkzTyH7CP7oDOYHAY1ae9Z3J2wgk6LoF+TR4cQTSEjIyMrlYy7ujuL'
    'ZVmoMMHungiC25KHnD7R9SruTm05hzU34a5OwPQODUP0J1N+l3c+2YElkb2UykpEMd0AHwpI1TtebxvJLsMGGeFBOkWqpg0sjmPi'
    'WqWdF7OsY73GZAgqMclQK3MeVkqy3rdq5hvD0dGXCqk8hYWGnHrt773lyTZTLJkPsGbDkxVMTeucCQfg9JBiofSyXRWiWyF4gWRo'
    'OlX5JhrjyZoRk9uGCBMS1SmV615zlnLH631Tj9J9KLQVmCIBIh1DKWMbpmO2bH+yDUMpvIPHrDA+hLaCaXWePXaMTg+drXDqQkgN'
    'NjW0zdX4V2foKKWDdQt3bIv8/XDmS+YDZf5q4dY98Q7a6SF7aNKpdcJ8gQFtnBRNC8ep3rnV9GQNfWZyuKzxJjOgGhNW+gh750B5'
    'EwZJrlKzdF6uBJiJHMjhmT/F4fzOfkDdy/xpu7UUEyvWoHdnuUfZYf02Rs2WvBfSBwofZCGWCbMx59lnx/dbuO5RpFXNOGvsGo7A'
    'IzBmy+4ksSns8NGVrRZelB6l4iSRzte6I+5XeO4Uk5A5aUf53mR2Ys6hfTzC3r0gj0kQJs8lwyrTPwnDAFqtVkkf4fmEIdrlOXXc'
    '9GYhm7COK/4e5c7P2xdEmcnOQHaAJEJpCl/l4Suh4bhJ8HSEpbkc68SYCbB1XkzVNRTuaJyZAbrI/Sdzd3KdcMNLvk54sMBu9LMa'
    'Zgi2sLncnI2rhEe06XhZAUkI9S3sDWykWhwR6fAMuGvsC2SUPi0ndiZoxmALT4ywPAuGO5hBharV+ert7KCZA7rUjK7JKmlPCOZk'
    'VzEl0loF2WedzSBsYcIkDQMF+uLrBe6x0njfc6TMJNClwr/TVnC3BlLQZES7+e6gGQULO8UoUyeYorPf4RqLz0A6nK6EvYPqooUD'
    'e4TUTl4Fs6R4ZWA5G4H2rldIT7hi8KU2PRP3wmiYAxSsyOegmGmgi1Sy7EyegMSXFEqZThE3b84WLle4kIiA/SNib1PuVm4Eiwvn'
    'cYS9gxBmGEy2R3ajgqUJERqKPdMbRjxhQQOGIY8gYbh8qSeUuUzY8TlmIuhSY/OrxJMKQ/WXgd9XT54d4X6F9Qrcic60DkEnlyV7'
    'gYOqJyH0KX0XdUAEmi2V09n5+CGYhUom7dNMd9DzSVeUXeYIXrjLCTWSpC01oPER9hZ2UnCgpgIQJj4VWpNxUMio0GleI2rKwsKy'
    'JZPhCwUqOhVgcVwOSK9x/ZS5C/oI29g8ypZrWxQq6qhoheWHpZD31eUJhymsJF9FAlhOHYuhTxFsHmHv4aAsiMibRZA/ExTrYC3D'
    'RE7arYiJK63DBL7pJNwJrYV7ZKWIFMcbtDdRXaS/xdnYRcpkyvkNTJqCEclf4f6EUQvZ6QXjIPuoz+FOZUziYVlTHfSpACzgSoMs'
    'BTFkyDbYQMVLR0FMeWxpYhjY3U6MLc8zSZgrgMr1unkzH3SpThBA22HAF4AOP4BtlGc5AzEBsqU7yXzNkCyQEDsnU3NcmsGNV0yD'
    '0KVOF/0QIEE1tAFn7+qLRW7kSJcnDYMpkzpIsAvZ9xaudyAPclGaSRH6SMWyTY6coOL/hTJ2o0qwzbHqpka2NPhGk29YG2mivITp'
    'TR3ccp2cGRL6VL0S4lSh4Y5OZeaZgZwx7OYtj+pPHBOhmQka6eXaGq0SYJ5cyFUp8yX0kXWDV9cbF6GlnKMCWmIhhczjZOAlPo94'
    'dZ6Qgz1QNDl0xQUkWhpZ03KO+9xFWRMkAwkLUuIVJN+dPIcfqWudVVzpiVdTF8KvbDFwAe8M2OnYyyPuDV0mzJkv+BIbAuqd9oOC'
    'mpHv/ptRsTgngfSAzKSDbBATQxU0Tc/1mlehkxYOLyLj7hwYC6J42NHQYDxHPJ64zr0GA9UIGtN/nfXTl0G1R3PNsdCJ1BX5jNoD'
    '41g4TKQ2cMeo8xHvT5zUokyLTAhT1SwqtIcgpMKhJ+5ddVp+OcmhQNdpIAgAgp8mTJrLdQ4mXbZ4p9RDC8I5oLhJX1qO0AToLrw2'
    '80IntTqnsbLxorZLyKFR3qaFjM3bf5zpPuIVFDKICJz9TrM2bM4k5DpHqaZdmJfAWqa2h0zZIK5scsE4A+nxrPWW95UrXtE/EiwD'
    'BRbc0z+bMxHiOExENRlDp3BBgdQsO0TQmYAjTAITcDgHxFcTMls8ePvU0nTnFWX6nPUNgNsh7apJGTqFE/xS92yOOTAVpGuCoeQ3'
    '9v7EE29fIecALabhFgpJcDdxynI+1eQMneQknCZSWFhcAEaIDOBMQzudr/h84p1quGRMBXKtmGsBhDz5Yz3iriakGqhTrCFFC/x4'
    'lqLJLGU4Vx36mc42mafZ4nw6oCgYuyYg80vw0jWZU1n6qeZq6IQhT+ilTqs55NAegERwiv0QK9V8jcW7p1oU26B2Au4jky8TNiSA'
    'aUfcu8qoB9VIUoVCWjB3Zk4r0WKLE7xU0zZbvJmL1lmrTolBB/VdWgQzdcS9q5laThJlArMZxCZfIc9ONnIJLKQLw+1Kjji5cH1V'
    'LhwSbxUyUuTVJ4T1XRnvKsVQk2o3bRYQ2ISX8CXVoxPnO+7KzCdO2YD0qew4u60FxpCDR7VjHnHvqjrF6ntvayXE0jsVasmEEziV'
    'ks+umszZ4lSvSLczWTPKu2RStdzYjXYKFCXuXXVGy3HXIDM0oAup0AFUFKiSfOKIalJny6ddseMUjqKaAbgjSzDIYpzBl11zW8gA'
    'ocGUT7TNznZn/iFUUz6UaC3xxMktBcQujnnCCxLLQMbLEh7kWk3v0KuhV0x5AtcFtzQgqEO57CGkVc9OmeLZ8uYMBczgj/X+itcr'
    'X3lOxU/XdpjloZMwLvm8RFiXTIHKhOLNWiMXc8LvaqbH8pnCNqrbJpHmIjEfUmIByJ5PqePHqXbiBL2eGr4GYJB1K2bzXC4j4Cal'
    'mOd8m/DZ8tQ1asdIvX1dyaRwx0UCk9D6DH8XSEOj1uBICJgT/VNorHASCEOhhOzLfX958rCBlI4Mx7fwYQuckR1qH7xed0E1jG5g'
    'EaSXCoTLDhfbzCaBNfFydWcXWiOvDVQcJ0swXcs5dcBdBAo2HCdBL3nvrno185pOsDla45DDhmlZwRcHfNRdmG15zoVmQY2S5rvw'
    'EfiNDl1f33p6ewnGdeCoWpNmmHmVwRrhuiAZqscr113GbXnyRI3IC0ALrTtMsss1axVOOq7u6m5G0l08JRe8wLjwxYOKyWJvtMZZ'
    'n130jTzVCgTyiWKVollL5ZuJBZDzVTdTQ/QitCQ1GghAHeqUDNP6suR5XdNgdmjLU6jUKFBhPHK7+EY4YBnPOU9eqpohopferFPS'
    'yOc4iFvk608uR0DtrqdJoi3PnqM8ICKBW/k5s9ADsijuesYusNdMJ2Rco7rQ1Fzl7cIJhNeVFMOR71deDpXaeEgZyjCF/ufJRnTQ'
    '+5X3/gYjh+tQBNxIWurl2GMIw7DOntDW5ShH3udLYDIgVhTJkbsZJi7lma5+mjSqJkZAUXC1C+sjH9OpjjHbTsrtvN+80ZaHFtLL'
    'tHiW0oGOzcrIQZXrz00d0Ys0LRlwPrMrixpnGKikiG0eAruaPUKeDaqkpJ3rl2UWEIWvKZjTfErQP9UEEr3kVyBhZEMSrOSi9oPi'
    'IiE5KTHAb8vHk+8QIIWwdTqnJmjdHJAprL65sGoaiV6cPm4mCGSaJJfNnb4rok2givjop5mkLc8powaQiiN4bMKGgWZoPdsJS6vJ'
    'JHoJfbA+6rDM1bmgEnpSTpqjdscznzyJto6lRH+0WFqnYko+swp3vt5f9eq+7NCpml87ZCpOQCoux/FcuGZWyfKFeEMAx6WDlXTh'
    'pJSZ0kBp0gkiq4klevU6XGdIwhviWh5XCAzeVh5k3TxxNbe05QOtGGXnh2Wisq8+yLbKVv/kvb/q1QkYKS8kn4YGCh0triIJJ8gE'
    '3/NlhmnLU8HRiYAzHkdIxidNuqUP1nX8i0kmemlnqUrQEQ/fxWG5QKNBsVh959E8k+Wl9owHqEaQSJ3Xar46os/jkbf8vk/TqZf0'
    'HRyiSQgOebzpAnFZ6Ua248jPJ98pVEjy5is7TGTPoKuJ4C9RXc030at3ylyEO+EVYCJkh6gc1ro1MzTnek968ugn4aHr9oNoC8sV'
    'UFYy1Meem3SiVx9YXamPkDvxEFlpxXzU/lR4+WMfzDtteYIJ6gRIQFWIOJKmX9dbzXT319QTvfRsUKiSdqEUlaNOKRFHacuuvzb5'
    'ZPlB0kNGvRHvaB8FV0kKU1z8amA/1fwTvYQSKCzVBKi0g80VkhT6VbBILH/PrymoLU/JF56Zqlw9JxilHJXczXz2xCQUvfoiAyFN'
    '5A9ERUlWBwsDeMg3hK6mobY8lMrQBsjBmLDZuddOtDVugszF27BQ0gTX4GDXKnQmhWtUY3fhIYKpMx5zUVs+6ZxQ17M478KoQk6g'
    'RuqEtedHP01HGd9TZwDVRm5VWr+glXU6BJb0gXTRqgkpyycolwFNT82d/MCcIAoKzce8xQzVlBS9CAk7hf9y9I0IVEAIAkSKKt04'
    'Ne4fl0wfeVkojYdC4WZ+QpGQ5qttgJ292N+0FL24KkP2WFFB8j0t8wZcmFpAoBujmZfa8lA8gEl8s+LFrkm6gLkMqc+1h2am6DUw'
    'aYQ1irWkZdorLlBQSNFWOCI8HebrICv5nbhaaHB2oBE3DNLYt4z440oq7kx8wYzUEmmCXPDTzhHGhOD8Iu44rHhd6coTbWtAslcO'
    '2LQs0pD1VXAnINTuBpieohcpc+5OyP4McD1XD2ARqH0RiL0xr/mpLU9EN7v0jJRTJ/UDHNGWaAbngtqnmqCi13BZl7yX67ng38Eq'
    'yzfJYPWvxzZFtTtgp2W8NWdWNFGcSmGzNlpaegNZc1T0klcmykVPp6MlQd3AHmhk0OqneqSapdodqESdivGBsfAI1OYQAlSX494z'
    'YJ6KbjqBlAUJoDo1WrCOmz6rLua4RtpMlTtIWxQR6eOAgiBN0ig7WAP+5tzq+lRTVfQiFcJtoATcFvjulIS50HAosiGzcW5OptvB'
    'Nm0VyiynF7XLqg9XBFObdNxMpH2lEgPkdI5sV7OxlZKbCczgvtRuiU2k8jogqDjb5A/gXEeTGFZ/6sbIp4OvYKrbsNpzfYVSNjYd'
    'kh6+qNolH67FpUungy8hEm8sx3GUNe5S71zIDs3TwVc2E1cz+S8AjsoQxREoAyyVDnTFTZfTob8OSWoo3GQMSybzlD9kV7Zd8BSm'
    'rug2udHyXRRxF0AyNSPRXdhM/XF9HebtwCGkoFvqDSEmNLVI1SiQnCQZ1t0HXwnlHJddDkaBpcszFSyVcsoaddDPRdlPmMFyD6qc'
    'NKbe2RZToxDIJF/kgUo+CD/MYdFNx599UnRh/r0TYFWnwWUCBND6ocvDNNbu4ZKbBOnikEWRxi7tUoQlyDjr7eHNpjwPSpiqcfll'
    '4L6ssAJ2QJmsH7TKuD3i9XCMTLgHiaqQtXOJM1y7OPWSc+rCfBYNk1SsuX4ZExwgCRAXpw4dak3y+OYwpbV7uFg9yb4DGii6VVjG'
    '1DR9jnu63/CG43ea0x2yc+vATir+gMuQMVLbdbdj3h6Bd5JtwaNrA4nQpFXWQuqObv2Xi+ggMWW3XCelARQyjfAfkMyapg4IepDP'
    'Wpnecg8zDBoVZboU8midp8swtBCyV+3uoBku3/k200dZPfEdoEPYMahpoIJirYvDwhzX7uHbXXYiXt0KRiSM4J4c9aP19Nh3rLHI'
    'ROLZ6Z9wBW6lBtwXqioZ6XF7xOsBkZt9b0A9+uBSrKG1NFEGqF8tMddFv+kCzMw1LKDggNUx3GDppFY3XeIFPz1sZKRz1HdywZDA'
    'JHxTVpClXOMcJrzop612UavwJgXAyAh8b8piUCtwtcSU1+7h2xbZmRyKpzWLRV1SIX9CCcb9hve8cHXFJLGLXjGmMlWjO0xd5OdW'
    'ObjPF9/cA+6VMkVSx15dCp9IzkFcgF1PZOUbc1/6EdAls4Ymfgd+MJxtgQto49zh/oS5L/dI03fM5QCxM5o5Sgstp67Ok56Zm/2i'
    '3wKjkl8tmbrewS1pAy+SS3JEI+6o4vaAjWVU3F3gG2S9yCNCCGmL+qGKwwwY/bQY05eJOOrJFjL7omfl+gwhxJ1Hfz0a8bR0O6je'
    'YmUbdUyUagzun6zbw3uufpwb12AOyqwwkfAUAOBEZVA991HVY74eu2qzkI5u+M/BtkFR6OjD3d8d9J5zXSlcHUXVAGeVO3mAJDPW'
    'k+Kfg93DXNju4cReYds49ik7oRLkoxcV1Xfm+5cRwg7cayVLh14JhFcSJ766Sx3hPFW1sX8ywT18K6fw0wEDbyKElzjnUA9yaONk'
    'FGP/loKvQpmtoEaJxMjkFtMakBuUgiowT3dU8XpQREklaA+wIWV3yekxbtuSfDjW5/z6gnZwuFKscg1tgJQKxXAEwI3SwXIrueL8'
    'LEMYWPjmGtXBEzO8SJQPsg2oy6XlY/9eA1n56euH5CCF4KU0yzlAV/cH9wEOdI79Qw702Bf9ierlYhziBOXo1PGRdKwn6R37Fx6c'
    'Qrc7r5w2eb/Fpa5B8EeppwKec8/+E/uXH+gxQQqQQCyvjn0jAhsOPijp6xcAmCGj31o7fzmaa0+kfnPWnW3V0RLqOpxRmCPbPZrv'
    'uzTCLzL3qUJOmqCQO6yX9XVNy9f9ku92ZZMVjTvPBG2UNAW1/TKPV01MlJ0u4cznyL70JnisA+KMqQ4kxPQ462uuzB2T71jJsAUV'
    'S8yMWqbKBetKQXC922667HQx96FNzJgOLRdogVukQjVcI7jI1YyZO2or/JVKRely9TvHnlzj2NWW6faZvz4uyQqEZRa0QtJBzG9Q'
    '/qDdOkRDmDhzR50iVx8LU9utQqhWM/AN8oGKvLMxJs9On31TRxoJ6b0me0/tX1C0LB2eZ83Mn238E7F/DKP4UokLAjgulaTK5Or6'
    '9SXm0E4fNF5DSyyrkR1ZiOVYBOb23HH/+PrC1z2lS9XXg7g7231DPPcK+ShoTn4zjzudeH32bQK1dm7hGN0tasj44QWBzjpORYFx'
    'wNc9Uy+nvD9sZ+TyVvjHCKhgEKpL7fbpr08DZmeKOAycQCh9Z4nJwuxKt91n/1qL76Ls20Lgh+oLAFSvZFMdmgwF2nfd5uvTp+9k'
    'QhX5PouCEOHUAe3P9XLKWE8fq4GR4DS8a82lA84jCe5Xk+xgu7Xm/SGZ9PoM5+kplKQMlTsPOqCQC9yrV/CRLiY0z+aeifSCf/4k'
    'nP3KQCO4b2EefLGU/eiOubbdZ3IZU2OjYoW4h7LsRK2rDrmvgN35mG9zT03dV9iXc1h4cgFvcgq+cs4BPxlHx9i3Ty+7jCg2DpXZ'
    'nK6x0uLZyu8rNp8w7+aeQCdu04MC277FCH5ZfEeBYCIPefr0X5/WPbadWeI2JitH4aQMV8q+C7HHZj0Y1q/Z9/0tiGffcSxk+SGp'
    'ZBezf7dEQFl95q9Pc0qNQBF9M0lMakI6OqmCqCf7/v8BpKWEMTlMAAA='
  )),
  'results/n7_scan/noroom_off0.log': ('f21c3d07efb7c4a8f52d236e6b5eb8166e97ff772af67e630dc5bf94e118d32b', (
    'H4sIAAAAAAACA32US27TYBSF51nFXUBk3fcDKQtAQu0A2AEdVIJ20DJC7J3zu5CUiQdW5MT5fHy/c313cRl6uhS9PH//+fr4/PRy'
    'YXp6/vbw8oGEXJo6hZuEx1XDKZulRCNIZky8VYSYXh9/PFwmt5zT3cXVDpiq1FXNJKKAeVBJWwPnpFHG+L7sH1PEt7YdWkfQpB7P'
    'BU1OaaVCCpPCD2pdyhqcNyhvmQtqcgA1plk8QMfb0qkq3X0kSDMcaVlvSZW30h0aR1Cn0W4nURNOZWpllhYOMhA9ov0Gdd2qduiR'
    'J2taTw9PyKWKaxunGZFCJo3hhts7KP+F+pEoV5q0SBI4waSEuhFwCibMO73U+QYN3WYX5UeiPGlaFI9vNsYQNTpiq1OwOA56mF6h'
    'VVvLgsaRqICoGYYoq+jmosFHG3iActgExnOD9ta+Q49EoeyCri+qo6WNc9wEjyyYNiZXaIZZXanDm8ROPTIVa5cs8LRggottEPY1'
    'Z2ujiDC2qltYld6KFzaPXOXChBlkOeoVyxq3SZqxUxjmPiP1Dlub7WnzyFYuDNYTMwBcE+uytpNjV4guYBUk6voGwMptuvuqI1/F'
    '64XCjglHrWYubOPPjG5QTjO6EX2tgWZu0qfP95++fvl4f0eLH/TLuM4Y4dlDz4iKY85ehiN+n96u+T+DvM+AmyME9k9SsgQdElVe'
    '6z1GmZNQzzq49i1DzMZ1+gP37hVuMgUAAA=='
  )),
  'results/n7_scan/noroom_off1.log': ('b52f738d6658eb6ac1009bae0f9cf5eaff66a9cca2d71f31df7ab77c263e5c1f', (
    'H4sIAAAAAAACA32USW5bMRBE9z5Fn4DoeQigK/gU8cJAYi3s3D/Fj0TyiiuB/MBTs16RrzdXpo9b0ef915+v9/vH5w3r+8+3zx8k'
    '5NLU0SIkXMVqQRns2WpDMlY+EirE9PX+++02tlxfXsH0A1OVujyDRLiZHcwCK21v9Uy3YOPB9JV9MfvETOoRTzBdpN2pGERsCanw'
    '6JiL/mcK81LbUNMD1JiGOxjQSlCLylXSgSZNd50KyQdUag1f0DxBnUYjMKnyeAgmzUaIOUkaYOfY2HdoxYb6SZM1DQKCE3Uo2dZY'
    'nL3aSSsRaac/oZpL64KePLnSJFsBCiOOodr3gEiPDF/DmNMfUO/VV6Z+EgVFUzUolMFyFaCFoaVg3HTw67D2hOaKCxonUVA0E9mA'
    'Bk4dRiNsPQz7lt7SNvk8fuoSv6AnUeEovdhFnT3s0ERJd2TtxE0ZKcSD2rnmCjVPpgI4NinA0S/AgqZHu9RB1ZpQ7D6b2r44L+pJ'
    'VSqojgxARZEm9+yKU3viUnlWW1jlc1jcKrtqlSdZiZZyNpojgUs0vdfNbamlFIVsxRrN/IdVndWysXXSVbwx6DmwwVCDTERsPzDo'
    'XOCv1HDeb1i8Ade7UidhO1Ph2EMHioTLiXV0V8MbxcTgTjjXA2u1jF/+AuvVx6YDBQAA'
  )),
  'results/n7_scan/noroom_off2.log': ('02c3068945f95363f798d48d4180805d729917821694211549cfdc5beb0a2856', (
    'H4sIAAAAAAACA32UTWrcQBCF9z5FHWAQ9f8TmAMEgr1IcoN4YUjshZ1VyN3zWnZmHAhi6JGEWh+v6r3q27Or0OO56Pnp+8+Xh6fH'
    '5zPT49O3++cPJOTK1DnOJCIcHUU5ETHuSsqjNRPtxPTy8OP+LCyb1M0toHEEdepOXVBwmI1KW800g1QD711krtDYTHfoHEGbhk0U'
    '0OpIZqqsqGxx0glPHW25QLW27gU1O4Ca0ihLkih+DVE15iblUFrNNl3e76EhO7SOoEljtXqqXlas1NoxGtNkDGwZG1+g5lv7gvqR'
    'UcChSFgh2i6r/C4BLUGyahQRbdeeRm9jO/TIKHea0qXUlFXL0GLVzA4nixErVZR6gdaWs0OPjEK/Zpgb0KglEN2AdHSiyKYys1iu'
    '5Re/RSqOjAr4DiYvqROIqNB0S7cist6Je2Ts4hSq+Us9cipgPBQuKnKCoA+eZZZpnOQllazBcRE7vcnuVR55leDxSj6whViuXnCi'
    'Ul3diOXg+pOr2t46d+yRW+kLo+sSGCxb6kUwq2MYjkQ6G7XERa26br6HII/8yqWuZYmMsBKMNua2nRFiqJ1WDOv0xTG13nwfrTpy'
    'rJZjA7+BnVAu8GUCZ4qIUSYv69wuZ4tGbhY3n+8+ff3y8e6WFr/ol3GfXOrkESfUcfJyrMSq3//ZjA2Cl4FNiQ9LsOx18+uefwXr'
    'e8F7O3u1N63RgDVrgakQRQIKh08LbpP0TfDEGoc/0cBhu2IFAAA='
  )),
  'results/n7_scan/noroom_off3.log': ('998ffcc369ba61ee093d7a6837a09cafc74e1127be9b3b5ac0a941e895552dab', (
    'H4sIAAAAAAACA32UPW4UURCE8z1FH2A16v8fpD0AErID4AY4QAI7sIkQd6fe2OyaZILVSD2rb+pVVb+7i6vS46Xo+enHr5fvT4/P'
    'F6bHp28Pzx9IyJWpMzVIeLjLijKsLC2apNm9LNyI6eX7z4dLzzZ1ugMzj5hO3cZgivpoKOVItrMzKSblahi+MYVjM1lQ4yNoU8+M'
    'ABoV7kElIS6cQyplgwNk36C2me9QP4CaEpSNAjqipkXlyTBAoNQabyJrrlDRLXqH9hE0aWwZK6plLlDaXTragE50SkbKFWqy6Q71'
    'o5xg3YTEgqbAQriBB/PKSbvHuPCVK1QR1Cv0KCh3mhw2QKcyHVCfaUsuMklpYeUb1HXzHRpHQYEynQ2oGaSaILconL6ErB0jzq4r'
    'NG1j3qFHQaEuwuxRoJbMLDvU4WmUk1W2Yli3TmVsnTv1KKlIUFVX1VEkECA9TdVWqTCJdYS4RdW81a41j6JKhISqNyTDKchdXxEV'
    'Q3+LfExTx8b/YZV1m30B8iisdGACSwrs6lOsncV+wtAM8g4PQ4XfYWXzWdg6igs7I+gOCimhA/LCdvWwSFGomaMEcjUBu7axnD7f'
    'f/r65eP9HS2+029jPTvPGYLOHnhm4JdnvPxzev3P/xrknYZa6Q5cgYZCrZGuIJFqxjpSaonBJJgnbxo8N9/voTqKt2A8bovV7xRB'
    'C9eNhJ7brAsP9U7XsOHr0dw35dNf/Qq5dDMFAAA='
  )),
  'verify/g3verify_rs/Cargo.toml': ('13cce447b589eca76f03cac868a1377d9a308c5804c201ef4f1a592b70e25b07', (
    'H4sIAAAAAAACAy2MwQrDIBBE7/sVwXvFJOd+SchhG0dZulXRNNC/7xZ6mjcPZrbGx5Mzdir8wnSfXF4vdEkfR5ZDavnJ4GcfHCHK'
    '+TdLWGZHtEU0lIhyCMZuvfWaROE7FDzst7bzprigtlop4vHORol1gL4SeLRrfgAAAA=='
  )),
  'verify/g3verify_rs/Cargo.lock': ('6bc640003c7b8438301e91588404fbb3eb1b1279ee8af6f71819b957fe8a1439', (
    'H4sIAAAAAAACA03KMQvCMBCG4f1+xZHuoaKrIDi5u5UOZ3uJh8lF0quQf2/cnF74vmfA+1M2DJIYe2m3kslkoZQaXiIrVzJe8dHw'
    'SjUWDwPe7Ee19Kixrv0OpWIm3Skhr2Ki0cOH6yZF8YwngGl60/KiyPMMSpn76uKxEwnN/VE3+oMfHXwBO0AN/JgAAAA='
  )),
  'verify/g3verify_rs/.cargo/config.toml': ('8ea425180afe0e2fc0cd930d9a5dbd4da07aebe14263eeec4d6df9c7ab996a20', (
    'H4sIAAAAAAACA4tOKs3MSYnlKiotLknLSUwvVrBViFbSdVbSUVAqSSxKTy3RTS4otc1LLMksS1WK5QIA9Q1fbDAAAAA='
  )),
  'verify/g3verify_rs/src/main.rs': ('43327a31015802e3eb290b7842d22a55d736db850500067061f1ae8f8321090d', (
    'H4sIAAAAAAACA71afW/aSBr/P59ikpUiuwEHSDe7S0Kk1bY6rVTR1XavdxLikLGH4MbYrscOZBO++/2eZ8ZvYJL09u6qBrA987y/'
    'j8/Pj8Xtxb1Mg8WD6HZFEPkykfiIMpHKbrBKQrnChZsFcSTihciWUsjN0s1VFtxLoaSbekuxiFNxO7uwIvvoHBCt96kfK/FbGs+x'
    'XXz3Y/+Hjvj4/tdP4ueLn376YdCzO2KdBlkmI0BI3NTNZPggFmm8Eir1zm8vNFzHE+fmxsJV2cDxHILPOH72V4FSARAARDYUP4uR'
    'eHRnfcdxZ9FWJLEKmMIgyuStTJVQOQjNlm6GX6tZIDz8ufg7HokeMyAhhgcRxdGfMo0Zh4fd4rE76HT7nV6n3xls/xUJYcmveXDv'
    'hhALiCaBXOA2gCoDWWrIHSF5f7GzI9xUCj+A5CIvsytW3gWLhUxl5EmlZUAwfxFJGt+mLqBavgyDuSQpdZjSSkuetIcMQ4g3hTLi'
    '1JfpkIGs3E2wyldiLBZBqrIO3Yz4SSpXbhAF0a2QWsOKaH33/pff3//86dfx3zSUqxJ2FIuTNI5XJyArp31DIy+30sNdV+VzKIPs'
    'ZAJFjKda3l4MAwoiBTICJe4DaEb6HaFiosRgECKRaRdsZUusz4mez7M7a2yz0DwXagk8NxRW3UKBp86zXZHrlyIl64BisjiVvnDx'
    '62G1klkaeGINUPFaiUk3dUEOPqa4B/z4JbxlrCAqECVCMBpWoAvDhbh+hxfoLUkIBsUaZIgwjhPoDAL77f0//+iQ5DQm4A9hxRpe'
    'Tf0f8yzJcTuIJOMbD4U4GY+uxzciGl1HN+LzELLo428gHMfBN6wtDnNySTW6xuXNCQNKwlyJGFBOPn388Pc/fv04Pqmg1vQUdUlJ'
    'THdpIBU5uXJv5bCKCpEYh7EYLwMxUZlMpmISLxYAgB8rAIf1zAB/FYShnJbKxD9r96m4gaOlUpHwoZLS7mJYQUqWAkwii8XNSCwg'
    'xNQav0lW5/1er2dfCS+Nlep6S+ndEY/hg310lCtoN/OHQxndX1VXQTwc/gNKkldHR0CWe5n4pK3k8YgIi4bgMfhTdvhqHtzOcCe4'
    'fKuvz89xKyOjGbKfbRBXdDBBADyDyS2EBTPCGv0YtNG9EVsNDJK1C0at0E1vwSzdD3JlM3CsrKH6buKGYbyGe7v+zIt9aU/5PpmR'
    'ahDJQHHrs/Su6S+/fHtzox/5QZo96CeW3qE32jcd4oVNMnWjW1kSpyahccyV+yDmsgx5BA7UamCgEhCEEYlmQkRS+uRGxojFB2F9'
    'KFyl0KfmlK5UDRLf1I6t7+blXdhyjbWgYs0YkJaYpgMCO2A4K/JFMAPrMfuE1YdaqmRlH22PjiijNc1hEZEyrVMlw0VHhPdhIXmR'
    'MGZbdG/EPI5Ds17rIxNfAdxKYBK00YFmbQovvPWqXGjxw0Ls9+F08lXc3IjLKX1aX8WpuLywbXyB1JHo8z5QqaV+LqqNZ/0p8NUA'
    'ATEyizjruvQxcLcdiHeFQAL9yI3rUW6CxdaDm0UmOtL04qcBa3dKdF4YJAn2ZxyZBXwHLpUnSZxmADR4g/TWEfw1vRJkMtCFkmto'
    'QTqFLOUmcSPfOl0hpO2L1DXKBAgj3KZQU6KPnNhbJcMhFGnVqQXP/SkooFyXr+yrxtYwnpFTQind9AWt0PJlUCx/zWoAX5P8NQ7S'
    '4B40em6A7j+3UMXMUOggBfsq4192oQqtVEclYZDN3GwGyVmG2R0WAWQoTidwHTKG0wIm20NzJZBgJemgXE0XBe5Jr7YeimddrhES'
    'KKez7pNU3gcxuTziEjl8tkTu1l4fygVyb56pAI9ocSTXJsc1mfbDGPwug5JVDlalJisSuKihjIodjtM0ABI9Q2GB1OyFo5/KJgGx'
    '16uAbffB1iC6G4uVBWAdQmc7zoiAfwvcFl7IkDSlBN1uSNdNU/dBcSGTuD4FUM69iFSD8fnlW+wfGA1Q4CevlS7CE4mXqyQKeioM'
    'qDycSyQMhgSuOKw7dUxENklwtGTiYSDiiT6vr1394+bGLe4MylsDhBCf6gbY/oJq5jkge8vuIpVSIwbdSklV4dKIZmtGNVtPHS9O'
    'HmZUvM54g0XG2VhhN7W9IgZQI44G7G074mdjX4KDFVzdbfHJ0sTWqiPmigzMwgZyvA7/4sDKGy8G9v4+FO/aXdeiq50bStxf5tcc'
    'p8FNcykkn8VrF5l2GdxSTioKBuQ1FDx5mHG9kBiV6N9dcLiHME96xrMZHRatleOUP0ElKN/BHiwgAUofvR0pFrK2/PsOYPZs9jAH'
    'lVHKQcZ2/gwSCyj5lmXbLfvp3xv/XjyB9N7V3uNt486W80ELFM1af481fPTr7OGylcWSk5IV+uy/yJC57r+OQUv1xPU1m9MTLvqc'
    'o+GfXbplv8j8AZOAw77SIs7aLMKPmhZxVlnE2f/IIoDy/2IRftTfY438sMnef8EiagyZ62+wCBhBZREwj//EIrb1IO2FKD+5ySiT'
    '6EQXFh1TQEyJC67A1jFKzzzyXXTanB12q5KVq+5mDKusTij0VaTBFhrLjvetoghuU3E6EseW1UfJQJzW99nkm3ZbMjTlz52UZM2W'
    'KYF0/G0EVVBSrLsWwLBPBIdWIqIkwWxg7LvIUSeXlauW5qJRIKNoKb0OFQAXX/x1hsyPRrq4urwQFmU/ki+ap8gNg9sISTpFW2YX'
    'ZS09RHHW1iYwlKpVyBuslZ0Cr3pFpWmhhJhzPvtapLOvh7JZmaJ2u4yG0OcmEgh/AukWHokYQJdk3WzcuNKFTGXi8I7tbkfyoZxg'
    'cBEDA0YXCVcKIDVF46R3s8eo29/q+ZOFn3YXtY7p2ZDLl+Dii+7VkINdz5MJWpYSvqWnQkVTSYOKQM8K1KTnOIAHxUZ+Q9EjjXSw'
    'pTkBtZhmNLUhN5pA2S52lRiqOeCGIQ3wdR8HfkF5BW6vxSqbHJrjzJiE5xud3fYmKgrhqKlEgjF3leRNxRrTxlYr18sglLxOXI8g'
    'SsqWjwdKm2YNTS0Xrae99l7tgjg6D1CXRpONWUMy7ZPt0I0vb1yeDXK8VfFKii97SIkBAjNk82+UzUWk/kJAugMwSFVfe2Kac1Yh'
    '5gt/g0w7mmUQQsXgDvnbdlKockUsI5B7qZEE1BKAmATaV48/WLsTe/aRarXw1uP2lEtkUatJa5wsdYMQnjKjkl9Z7NYQytVBYvh7'
    'j4YCLs2nWD5nImtfQQKJ77CM+vn2LPqibozoWDU0LWG1UB++KfVyYBePvQj7wkXcuTq4Zo5we9f+eHv08h0QByztJDDVevY0iabi'
    'bNQmy3KhjjURSZzcYNO+tBBsbcZV+K0GgFg1dbJ4di89yz6AzlFxmh18SrBoNuYkuVpa6hsLUG0TI1G3rEY0RxhL8+hQ/KL+/5lZ'
    'Dc2m0PmORJSv5ihvkXp3xoBXdAN3zIBgk5U5YAPvI/B0ruFB65w29AC/0cz+gugc+EguunVeUB0k5g9CeW7EJxecJ+LiiIpoeDcj'
    'qnjmOdcFk4jv69V3HT5lBqKjSzMlPneh7o2GXyjZr3QPzm051TMbSkP69EQqmrGR2yB1RHFGsZHwOo2ai6RTBPsdu0xllqfRoVqq'
    'mmPVs8A3TbRYurqXem4TyCxXXjdyzrcS/JdmY2SA66Ib3w2GOriuaVjCePTzlrS3DgBi/cyooJ4X9qay62C/hSKIhqr2kP5yyVuD'
    'dqjuPZh6Wure1/h9STiL63CKq9X5Vq1xsHcL/b+S71BZd3XSQ8vj7+Q8rSZyRB6aqIyLPaKDzyqfIbygO7MPp0QLQngjeLiWFSmW'
    'Ri7GHVpV1JLdDmU1OtIMorZ0up+a6umnHBe2J6Fa/jCV7aaVUgPmYIT5hiyl9//vU5VuO0raBy/Srn20KrRZKZvDoJ+BYs4kNARO'
    'aKBg8xwzlBmNkF/Y8pwvrkV31Na3IgMj+9IBvFUYGGnKTW+Noj5lKbyFdCWj++GQHlg2bCgMpVdqAbKkB/AtAnMt6pFFJgCQhdGx'
    'dfLq89yTGnfcPiRp7EmlhkO5gVMUjee2pLc4SAWZRMikP3USN1USpObROnWTglJeHMZFd8OLB88uXgaNxRfPLSYuitVNmdyQTDSE'
    't/sQqla4L7YVOC2NQwC/LwB+/xzAXgEQ0S1OqPJwQ1RkkV+ef9NpspUn9NIDD3jEOTeVXd1a7592D/kQX5+jo9AZl+SWp+yrQyRf'
    'FiRfvopkmvSn2bEV8QGFOD2FxaDd7A8aIvep8x9VB+76Tl0t3HYUi/DDCWPvbncBn75TGQmTPDOi1ytMs8vPgT5qHNE8cyI9bCs4'
    'lZsFahGgkqTic9zhxn8t9aExzXlQwX0QtZFAHZN1TQc1PFug81Uz5Phw1t+aA9aOSKdMiGIw+mZ3ADTp2WA8dV5HNm1FjcnU1IcT'
    'FpNsQyAoPrmmLmrfOpNXfIdMqI6tGs1IMzIRUXdAaNCMxTVcuss3GPUF43W1uJ6+PJESBjDDJ9TSfPF2fIZ2sVGw/1X5FFahz6mR'
    'jo4nPaoGRbRzXkjnwWWHRola2wnSSqO6NSa8U8yavQPa+7a2tz/48dAhYkj9sUXzJwemlKcucTlT+RxR0XZQQ1t2CxZO4PzD5PyK'
    '1DZETNjOmSNJJHH9YsJ7SfRaVgnGRtjgGuftzlSwfCVl0iO8BYjmIn3cODIAaQufsBa1Ga6ooC0w7Bftipy78S5F8S/qNHtRorV5'
    'C9iaN5iYzk7Vrt96YUPQ1pCTNfDSaWEVzT3mdRhebfVMM6sZ5bq2dRO9ItG4Yd5e0Tjb9xTvsjy3pnyzZTiM5Nqym0/L11uap9Mo'
    'OLRFvqnFdmiBX4SqIGwbR8zmrSOKInRc9Njb7k2gVUaFb0uDhAemFepVr6c8IeIX7YGZPzcQ1pr76iVDyw3XfNJdvWsGdK7o0wtn'
    'dh2hKYfZf/sNUsyolx7seAo7NL0ls2Ns2FTVnqwO3lg/cW4tEQmZLgx7RuTma6fKwzoqB/sHFzVlPdOy3iGeZ6cURU4NuW0n3lm6'
    'V/8pc1DlrJCun+6fxD1V6oqf03HWTk1YehO9/0blHxJvp3ofUIxHj1vx+Pi43W5PSn4Ir/MlRjV60jmxd0usfQ69fSorjTrOCN3E'
    'NxO9QzDTGdHH56HAZ/W+42ONcPz3CsrFid0pTYFLnzZGqA5ZhGQobU9NJhhxSXlVFuv/Bg8Z5pAlLQAA'
  )),
}
ok = True
for path, (sha, parts) in FILES.items():
    data = gzip.decompress(base64.b64decode(''.join(parts)))
    good = hashlib.sha256(data).hexdigest() == sha
    ok &= good
    os.makedirs(os.path.dirname(os.path.join(ROOT, path)), exist_ok=True)
    open(os.path.join(ROOT, path), 'wb').write(data)
    print('%s  %s  %s' % ('ok ' if good else 'BAD', sha, path))
os.chdir(ROOT)
print('\nall files written and checked' if ok else '\nCHECKSUM MISMATCH')
!nproc; lscpu | grep -E 'Model name|Vendor ID'

## 2. Quick verification (~10–15 min)

In [ ]:
!python3 verify/verify_result.py quick --out /content/verify_run/quick

## 3. The whole critical range N = 419…478 (a few hours)
Logs go to Google Drive so they survive a disconnect. After a disconnect, run cell 1 and this cell again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/g3_7_verify_run'
!python3 verify/verify_result.py critical --out {OUT}/critical

## 4. (optional) Every N = 1…478

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/g3_7_verify_run'
!python3 verify/verify_result.py full --out {OUT}/full

## 5. (optional) The independent Rust implementation at N = 473 and 474

In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal > /dev/null
import os; os.environ['PATH'] += ':/root/.cargo/bin'
!python3 verify/verify_result.py quick --rust --out /content/verify_run/quick

## What the result means
* **quick** confirms the upper bound (the certificate is checked directly from the definition) and that the two
  decisive values N = 473 and 474 behave as claimed, with full count vectors matching the published table.
* **critical** (with Korsky's bound) or **full** (without it) re-establishes the lower bound: no admissible 7-set
  with maximum ≤ 473.
* If you post about the result, say what you ran. The Erdős Problems forum asks that claims be verified by a human
  and that AI assistance be disclosed.